In [ ]:
import torch

In [ ]:
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # I  (x^1)
[0.55, 0.87, 0.66], # am  (x^2)
[0.57, 0.85, 0.64], # learning (x^3)
[0.22, 0.13, 0.33], # LLM  (x^4)
[0.77, 0.25, 0.10], # from  (x^5)
[0.05, 0.70, 0.55]] # Peeyush (x^6)
)

In [ ]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [ ]:
# initialise 3 weighted metrics query, key, value
torch.manual_seed(123)
W_q = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_k = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_v = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
W_q, W_k, W_v

(Parameter containing:
 tensor([[0.2961, 0.5166],
         [0.2517, 0.6886],
         [0.0740, 0.8665]]),
 Parameter containing:
 tensor([[0.1366, 0.1025],
         [0.1841, 0.7264],
         [0.3153, 0.6871]]),
 Parameter containing:
 tensor([[0.0756, 0.1966],
         [0.3164, 0.4017],
         [0.1186, 0.8274]]))

In [ ]:
x_2

tensor([0.5500, 0.8700, 0.6600])

In [ ]:
# query, kye and value based on 2nd input

query_2 = x_2 @ W_q
key_2 = x_2 @ W_k
value_2 = x_2 @ W_v

In [ ]:
query_2, key_2, value_2

(tensor([0.4306, 1.4551]), tensor([0.4433, 1.1419]), tensor([0.3951, 1.0037]))

In [ ]:
#key and value for the intire input

keys = inputs @ W_k
values = inputs @ W_v

In [ ]:
keys.shape, values.shape

(torch.Size([6, 2]), torch.Size([6, 2]))

#1. Compute Attention Scores

In [ ]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.1580, 0.3437],
        [0.1827, 0.3292],
        [0.3091, 0.8915]])

In [ ]:
attn_score_22 = query_2 @ keys[1]

In [ ]:
attn_score_22

tensor(1.8524)

In [ ]:
# for intire input

attn_scores_2 = query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 0.5682, 0.5577, 1.4303])

#2. Compute Attention Weights

In [ ]:
d_k = keys.shape[1]
d_k

2

In [ ]:
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2)

tensor([0.1586, 0.2393, 0.2324, 0.0965, 0.0958, 0.1775])


#3 compute context vector


In [ ]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.2893, 0.8084])

# Final code

In [ ]:
import torch.nn as nn

class SelfAttention(nn.Module):

  #initialisation
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_q = nn.Parameter(torch.rand(d_in, d_out))
    self.W_k = nn.Parameter(torch.rand(d_in, d_out))
    self.W_v = nn.Parameter(torch.rand(d_in, d_out))

  #query, key value
  def forward(self, x):
    keys = x @ self.W_k
    queries = x @ self.W_q
    values = x @ self.W_v

    #attn score(omega)
    attn_scores = queries @ keys.T

    #attn weight
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    #context vector
    context_vec = attn_weights @ values

    return context_vec

In [ ]:
torch.manual_seed(123)
selfAtten = SelfAttention(d_in, d_out)
print(f"Context vecotrs: \n {selfAtten(inputs)}")

Context vecotrs: 
 tensor([[0.2797, 0.7868],
        [0.2893, 0.8084],
        [0.2888, 0.8074],
        [0.2642, 0.7512],
        [0.2697, 0.7643],
        [0.2771, 0.7809]], grad_fn=<MmBackward0>)


In [ ]:
import torch.nn as nn

class SelfAttentionV2(nn.Module):

  #initialisation
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_q = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_k = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_v = nn.Linear(d_in, d_out, bias=qkv_bias)

  #query, key value
  def forward(self, x):
    keys = self.W_k(x)
    queries = self.W_q(x)
    values = self.W_v(x)

    #attn score(omega)
    attn_scores = queries @ keys.T

    #attn weight
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

    #context vector
    context_vec = attn_weights @ values

    return context_vec

In [ ]:
torch.manual_seed(123)
selfAtten2 = SelfAttentionV2(d_in, d_out)
print(f"Context vecotrs: \n {selfAtten2(inputs)}")

Context vecotrs: 
 tensor([[-0.5049, -0.0677],
        [-0.5034, -0.0695],
        [-0.5033, -0.0695],
        [-0.4995, -0.0668],
        [-0.5010, -0.0677],
        [-0.4996, -0.0682]], grad_fn=<MmBackward0>)


In [ ]:
print(f"Context vecotrs: \n {selfAtten(inputs)}")


Context vecotrs: 
 tensor([[0.2797, 0.7868],
        [0.2893, 0.8084],
        [0.2888, 0.8074],
        [0.2642, 0.7512],
        [0.2697, 0.7643],
        [0.2771, 0.7809]], grad_fn=<MmBackward0>)


In [ ]:
selfAtten2.W_q.weight.T

tensor([[-0.2354,  0.2177],
        [ 0.0191, -0.4919],
        [-0.2867,  0.4232]], grad_fn=<PermuteBackward0>)

In [ ]:
selfAtten.W_q = torch.nn.Parameter(selfAtten2.W_q.weight.T)
selfAtten.W_k = torch.nn.Parameter(selfAtten2.W_k.weight.T)
selfAtten.W_v = torch.nn.Parameter(selfAtten2.W_v.weight.T)


In [ ]:
selfAtten(inputs)

tensor([[-0.5049, -0.0677],
        [-0.5034, -0.0695],
        [-0.5033, -0.0695],
        [-0.4995, -0.0668],
        [-0.5010, -0.0677],
        [-0.4996, -0.0682]], grad_fn=<MmBackward0>)

In [ ]:
print(f"Context vecotrs: \n {selfAtten2(inputs)}")

Context vecotrs: 
 tensor([[-0.5049, -0.0677],
        [-0.5034, -0.0695],
        [-0.5033, -0.0695],
        [-0.4995, -0.0668],
        [-0.5010, -0.0677],
        [-0.4996, -0.0682]], grad_fn=<MmBackward0>)
